In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import math
import os
import ws3.opt
import pickle
import numpy as np
from math import pi
import ipywidgets as widgets
from ipywidgets import interact
from util import generate_radar_chart, generate_subplots_radar_chart, create_grouped_bar_chart

In [2]:
#Equity Silver Data
equitysilver_max_hv= {
    "Scenarios": ["Baseline", "S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9"],
    # "Harvest Volume": [9.5385144e+07, 1.3344616e+08, 8.584663087e+07, 7.6308117e+07, 6.676960411e+07, 5.723109073e+07, 4.7692577e+07, 3.8154064e+07, 2.8615551e+07, 1.9077037e+07, 9.5385238e+06],
    "Net Emission": [-10187414, 44209174, -20190655, -37717399, -53571451, -69020879, -83747319, -96901488, -116527826, -127261518, -141630206],
    "Social Indicator": [100842, 130901, 90757, 80673, 70589, 60505, 50421, 40336, 30252, 20168, 10084],
    "Economic Indicator": [3438933, 4464020, 3095039, 2751146, 2407253, 2063360, 1719466, 1375573, 1031680, 687786, 343893],
    "Species Indicator": [0.7329, 0.7577, 0.7607, 0.7514, 0.7445, 0.7401, 0.7375, 0.7275, 0.7328, 0.7365, 0.7399],
    "Old Growth Indicator": [196384, 167126, 222922.6, 247449, 286878.4, 324882.5, 367419, 420221.3, 462754.7, 502643, 535463.1]
}

equitysilver_min_ha = {
    "Scenarios": ["Baseline", "S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9"],
    # "Harvest Area": [3.8189149e+05, None, 3.3325811e+05, 2.8743943e+05, 2.4404208e+05, 2.0333523e+05, 1.6452611e+05, 1.2681471e+05, 9.060671199e+04, 5.7354080e+04, 2.7180641e+04],
    "Net Emissions": [-27541849, -155624647, -42297895, -55475110, -69237782, -83282990, -96462861, -109037924, -121829519, -133928525, -144780424],
    "Social Indicator": [92157, None, 82941, 73725, 64510, 55294, 46078, 36862, 27647, 18431, 9215],
    "Economic Indicator": [3142756, None, 2828481, 2514205, 2199929, 1885654, 1571378, 1257102, 942827, 628551, 314275],
    "Species Indicator": [0.7543, 0.7457, 0.7534, 0.7491, 0.7466, 0.7443, 0.7456, 0.7431, 0.7487, 0.7489, 0.7502],
    "Old Growth Indicator": [303678.5, 580504.3, 334075.7, 364650.4, 393684.8, 423066.2, 451206.5, 479354.5, 505137.4, 530289.9, 556633.9]
}


equitysilver_max_st = {
    "Scenarios": ["Baseline", "S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9"],
    # "Carbon Stock": [1.993899100e+08, 2.213324476e+08, 2.023201210e+08, 2.049510923e+08, 2.073555872e+08,
    #                  2.0959373e+08, 2.1161157e+08, 2.1346753e+08, 2.1511490e+08, 2.1654530e+08, 2.1772600e+08],
    "Net Emissions": [-34255783, -155624647, -49672695, -64270430, -78526028, -91980728, -104984288, -117144321,
                      -128555316, -139248380, -148466128],
    "Social Indicator": [92157, None, 83272, 74497, 65185, 55858, 46351, 37113, 27854, 18498, 9293],
    "Economic Indicator": [3142756, None, 2839766, 2540532, 2222965, 1904896, 1580700, 1265647, 949882,
                                630844, 316931],
    "Species Indicator": [0.7534, 0.7457, 0.7507, 0.7484, 0.7441, 0.7440, 0.7438, 0.7428, 0.7419, 0.7425,
                                     0.7429],
    "Old Growth Indicator": [317668.7, 580504.3, 343022.16, 366396.3, 390767.51, 417504.07, 445561.57,
                                            473603.09, 500597.34, 526910.75, 552527.14]
}

equitysilver_min_em = {
    "Scenarios": ["Baseline", "S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9"],
    # "Net Emission": [-8.4556995e+07, -1.6051199e+08, -9.6237592e+07, -1.0688514e+08, -1.1650972e+08,
    #                  -1.2533166e+08, -1.3318654e+08, -1.4030201e+08, -1.466282322e+08, -1.5177528e+08,
    #                  -1.549826706e+08],
    "Net Emissions": [-33180983, -154948627, -48819211, -63980807, -78384629, -92468485, -105856916,
                      -118833104, -130275827, -140664223, -148892042],
    "Social Indicator": [92157, 1169, 82941, 73725, 64510, 55294, 46078, 36862, 27647, 18443, 9408],
    "Economic Indicator": [3142756, 39891, 2828481, 2514205, 2199929, 1885654, 1571378, 1257102, 942827,
                                628976, 320857],
    "Species Indicator": [0.7258, 0.7446, 0.7137, 0.7063, 0.6987, 0.6929, 0.6930, 0.6971, 0.7071, 0.7228,
                                     0.7373],
    "Old Growth Indicator": [214539.2, 574569.2, 242967.94, 275445.01, 311282.5, 349133.9, 384958.37,
                                            422547.95, 459147.55, 498725.47, 538769.88]
}

In [3]:
#Golden Bear Data
goldenbear_max_hv = {
    "Scenarios": ["Baseline", "S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9"],
    # "Harvest Volume": [
    #     8.0053897e+06, 1.392895474e+07, 7.2048612e+06, 6.4043222e+06, 5.603783240e+06, 
    #     4.8032443e+06, 4.0027053e+06, 3.2021663e+06, 2.4016274e+06, 1.6010884e+06, 8.0054942e+05
    # ],
    "Net Emissions": [
        -22431846, -6207187, -20816108, -24435190, -27875150, -30827647, 
        -33795281, -37203091, -40172516, -43335597, -46257710
    ],
    "Social Indicator": [
        8463, 14725, 7617, 6770, 5924, 5078, 4231, 3385, 2539, 1692, 846
    ],
    "Economic Indicator": [
        288619, 502182, 259757, 230895, 202033, 173171, 144310, 115448, 86586, 57724, 28862
    ],
    "Species Indicator": [
        0.9228, 0.8086, 0.9170, 0.9198, 0.9176, 0.9157, 0.9167, 0.9165, 0.9198, 0.9224, 0.9262
    ],
    "Old Growth Indicator": [
        26625, 14652, 31478.4, 33438, 34875, 36645.55, 39003.57, 40695.13, 42677.27, 44921.4, 52275.86
    ]
}

goldenbear_min_ha = {
    "Scenarios": ["Baseline", "S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9"],
    # "Harvest Area": [
    #     6.0739280e+04, None, 5.2595413e+04, 4.4808782e+04, 3.7405442e+04, 
    #     3.0460783e+04, 2.3927580e+04, 1.790608433e+04, 1.2411201e+04, 7.4768617e+03, 3.1937985e+03
    # ],
    "Net Emissions": [
        -30666617, -49427321, -32934008, -35234780, -37362830, -39353227, 
        -41384955, -43112708, -44772679, -46464434, -48017286
    ],
    "Social Indicator": [
        7787, None, 6968, 6213, 5426, 4726, 3867, 3094, 2320, 1546, 780
    ],
    "Economic Indicator": [
        265554, None, 237650, 211883, 185047, 161180, 105539, 79128, 79128, 52752, 26607
    ],
    "Species Indicator": [
        0.8927, 0.9285, 0.8939, 0.8886, 0.8883, 0.8848, 0.8843, 0.8875, 0.8930, 0.8988, 0.9137
    ],
    "Old Growth Indicator": [
        27323.97, 48756.38, 28667.59, 30955.1, 33431.06, 35064.01, 
        37873.07, 40753.68, 43093.21, 45539.53, 47156.5
    ]
}


goldenbear_max_st = {
    "Scenarios": ["Baseline", "S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9"],
    # "Carbon Stock": [
    #     3.667303750e+07, 3.968485389e+07, 3.7097494e+07, 3.7500430e+07, 3.787936687e+07, 
    #     3.823291645e+07, 3.8564378e+07, 3.8868962e+07, 3.9134512e+07, 3.935881375e+07, 3.9544224e+07
    # ],
    "Net Emissions": [
        -32702620, -49427321, -34995663, -37138002, -39157846, -41090638, 
        -42923815, -44598486, -46225607, -47539676, -48576478
    ],
    "Social Indicator": [
        7734, None, 6961, 6187, 5414, 4640, 3867, 3093, 2320, 1546, 781
    ],
    "Economic Indicator": [
        263762, None, 237385, 211009, 184633, 158257, 131881, 105504, 79128, 52752, 26652
    ],
    "Species Indicator": [
        0.8181, 0.9285, 0.8223, 0.8284, 0.8493, 0.8639, 0.8693, 0.8780, 0.8963, 0.9092, 0.9208
    ],
    "Old Growth Indicator": [
        24711.02, 48756.02, 26270.53, 28295.18, 30210.93, 32173.28, 
        34377.33, 36712.57, 38970.18, 42030.01, 45547.13
    ]
}

goldenbear_min_em = {
    "Scenarios": ["Baseline", "S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9"],
    "Net Emissions": [
        -33380413, -49406956, -35592895, -37676137, -39670890, 
        -41552426, -43331349, -44975939, -46281647, -47578947, -48649924
    ],
    "Social Indicator": [
        7734, 24, 6961, 6187, 5414, 4640, 3867, 3093, 2320, 1595, 773
    ],
    "Economic Indicator": [
        263762, 850, 237385, 211009, 184633, 158257, 131881, 105504, 79128, 54409, 26376
    ],
    "Species Indicator": [
        0.8666, 0.9283, 0.8689, 0.8718, 0.8769, 0.8766, 0.8811, 0.8900, 0.8943, 0.9052, 0.9191
    ],
    "Old Growth Indicator": [
        19826.7, 48567.26, 24952.22, 23096.49, 25144.47, 
        27405.45, 30179.17, 33194.88, 36780.47, 39921.46, 43585.06
    ]
}

In [4]:
# Red Chris Data
redchris_max_hv = {
    "Scenarios": ["Baseline", "S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9"],
    # "Harvest Volume": [
    #     1.121138700e+07, 2.4993799e+07, 1.009025875e+07, 8.9691200e+06, 
    #     7.8479813e+06, 6.7268427e+06, 5.6057040e+06, 4.4845652e+06, 
    #     3.3634266e+06, 2.2422878e+06, 1.1211492e+06
    # ],
    "Net Emissions": [
        -46083355, -20553822, -50234031, -52652856, -56559028, 
        -61565532, -64192028, -68328829, -71715541, -76112747, -78934797
    ],
    "Social Indicator": [
        11852, 26423, 10667, 9482, 8296, 7111, 5926, 4741, 3555, 2370, 1185
    ],
    "Economic Indicator": [
        404205, 901104, 363785, 323364, 282944, 242523, 202103, 
        161682, 121262, 80841, 40420
    ],
    "Species Indicator": [
        0.7804, 0.7300, 0.7774, 0.7825, 0.7852, 0.7786, 0.7707, 
        0.7725, 0.7810, 0.7804, 0.7985
    ],
    "Old Growth Indicator": [
        100401, 47245, 104475, 106839.5, 110990.7, 114144.5, 
        115217.4, 119934.8, 124733.5, 128285.5, 139178.5
    ]
}


redchris_min_ha = {
    "Scenarios": ["Baseline", "S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9"],
    # "Harvest Area": [
    #     5.9129592e+04, None, 5.2317333e+04, 4.5728340e+04, 3.9326668e+04, 
    #     3.3093378e+04, 2.7023194e+04, 2.1148345e+04, 1.5497567e+04, 1.0023503e+04, 4.7586155e+03
    # ],
    "Net Emissions": [
        -60129917, -81552275, -62486615, -64832416, -67151265, -69383983, 
        -71593454, -73778154, -75962420, -77708010, -79628597
    ],
    "Social Indicator": [
        10831, None, 9748, 8665, 7582, 6499, 5449, 4345, 3262, 2172, 1089
    ],
    "Economic Indicator": [
        369393, None, 332454, 295514, 258575, 221636, 185826, 148189, 111260, 74096, 37169
    ],
    "Species Indicator": [
        0.8507, 0.8120, 0.8461, 0.8435, 0.8370, 0.8341, 0.8279, 0.8231, 0.8179, 0.8178, 0.8159
    ],
    "Old Growth Indicator": [
        111298.1, 150237.5, 116224.3, 119077.2, 122514.1, 126483.9, 129808.9, 133744.85, 138646.6, 142010.7, 146688.7
    ]
}

redchris_max_st = {
    "Scenarios": ["Baseline", "S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9"],
    # "Carbon Stock": [
    #     6.901783991e+07, 7.295730465e+07, 6.949017835e+07, 6.9953284e+07, 7.0406994e+07,
    #     7.0852103e+07, 7.1285194e+07, 7.170636811e+07, 7.2110418e+07, 7.2477947e+07, 7.2784730e+07
    # ],
    "Net Emissions": [
        -61347742, -81552275, -63710618, -66055037, -68411836, -70656314, -72753863, -74813593, 
        -76809883, -78787054, -80471375
    ],
    "Social Indicator": [
        10831, None, 9748, 8665, 7582, 6499, 5415, 4332, 3249, 2166, 1083
    ],
    "Economic Indicator": [
        369393, None, 332454, 295514, 258575, 221636, 184696, 147757, 110818, 73878, 36939
    ],
    "Species Indicator": [
        0.7704, 0.8120, 0.7760, 0.7818, 0.7842, 0.7849, 0.7853, 0.7894, 0.7919, 0.7934, 0.8039
    ],
    "Old Growth Indicator": [
        101872.9, 150237.5, 106999.7, 112276.2, 117122.4, 121883, 126727.6, 132072.5, 137455.3, 
        141725.8, 146174.8
    ]
}


redchris_min_em = {
    "Scenarios": ["Baseline", "S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9"],
    "Net Emissions": [
        -62345424, -81523017, -64602517, -66803615, -68914305,
        -70959314, -72963909, -75040719, -77005884, -78896293, -80492010
    ],
    "Social Indicator": [
        10831, 41, 9748, 8665, 7582, 6499, 5415, 4332, 3249, 2166, 1083
    ],
    "Economic Indicator": [
        369393, 1409, 332454, 295514, 258575, 221636, 184696, 147757, 110818, 73878, 36939
    ],
    "Species Indicator": [
        0.7207, 0.8116, 0.7425, 0.7510, 0.7538, 0.7541, 0.7507, 0.7685, 0.7828, 0.7918, 0.8011
    ],
    "Old Growth Indicator": [
        76629.06, 149966.1, 85185.72, 92755.32, 99524.74,
        104073.4, 111344.7, 121090.3, 130605.4, 138678.6, 145028.9
    ]
}


In [5]:
# Plot each objective mode seperately for each case study
case_studies = ['redchris', 'goldenbear', 'equitysilver']
obj_modes = ['Max_hv', 'Min_ha', 'Max_st', 'Min_em',]
data_sets = {
    "equitysilver_max_hv": equitysilver_max_hv,
    "equitysilver_min_em": equitysilver_min_em,
    "equitysilver_max_st": equitysilver_max_st,
    "equitysilver_min_ha": equitysilver_min_ha,
    "goldenbear_max_hv": goldenbear_max_hv,
    "goldenbear_min_em": goldenbear_min_em,
    "goldenbear_max_st": goldenbear_max_st,
    "goldenbear_min_ha": goldenbear_min_ha,
    "redchris_max_hv": redchris_max_hv,
    "redchris_min_em": redchris_min_em,
    "redchris_max_st": redchris_max_st,
    "redchris_min_ha": redchris_min_ha,
}


case_study_widget = widgets.Dropdown(
    options=case_studies,
    description="Case Study:",
    value='equitysilver'
)

obj_mode_widget = widgets.Dropdown(
    options=obj_modes,
    description="Objective Mode:",
    value='Max_hv'
)

def update_chart(case_study, obj_mode):
    # Construct variable name dynamically
    data_var_name = f"{case_study}_{obj_mode}".lower()
    try:
        # Dynamically access the dataset based on the constructed variable name
        data = globals()[data_var_name]
        generate_radar_chart(data=data, case_study=case_study, obj_mode=obj_mode)
    except KeyError:
        print(f"Dataset for {data_var_name} not found. Please ensure it is defined.")

interact(update_chart, case_study=case_study_widget, obj_mode=obj_mode_widget)

interactive(children=(Dropdown(description='Case Study:', index=2, options=('redchris', 'goldenbear', 'equitys…

<function __main__.update_chart(case_study, obj_mode)>

In [6]:
# Plot all objective modes for each case study
case_studies = ['redchris', 'goldenbear', 'equitysilver']
obj_modes = ['Max_hv', 'Min_ha', 'Max_st', 'Min_em']

data_sets = {
    "equitysilver_max_hv": equitysilver_max_hv,
    "equitysilver_min_em": equitysilver_min_em,
    "equitysilver_max_st": equitysilver_max_st,
    "equitysilver_min_ha": equitysilver_min_ha,
    "goldenbear_max_hv": goldenbear_max_hv,
    "goldenbear_min_em": goldenbear_min_em,
    "goldenbear_max_st": goldenbear_max_st,
    "goldenbear_min_ha": goldenbear_min_ha,
    "redchris_max_hv": redchris_max_hv,
    "redchris_min_em": redchris_min_em,
    "redchris_max_st": redchris_max_st,
    "redchris_min_ha": redchris_min_ha,
}

# Create the dropdown widget for user selection
case_study_widget = widgets.Dropdown(
    options=case_studies,
    description="Case Study:",
    value='equitysilver'
)

# Update function for generating subplots
def update_subplots_chart(case_study):
    generate_subplots_radar_chart(case_study, obj_modes, data_sets)

# Create the interactive widget
interact(update_subplots_chart, case_study=case_study_widget, obj_modes=obj_modes)


interactive(children=(Dropdown(description='Case Study:', index=2, options=('redchris', 'goldenbear', 'equitys…

<function __main__.update_subplots_chart(case_study)>

# Indicators

In [7]:
# Data dictionaries Equity Silver
equitysilver_net_emission = {
    "Scenarios": ["Baseline", "S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9"],
    "Max_hv": [-10187414, 44209174, -20190655, -37717399, -53571451, -69020879, -83747319, -96901488, -116527826, -127261518, -141630206],
    "Min_ha": [-27541849, -155624647, -42297895, -55475110, -69237782, -83282990, -96462861, -109037924, -121829519, -133928525, -144780424],
    "Max_st": [-34255783, -155624647, -49672695, -64270430, -78526028, -91980728, -104984288, -117144321, -128555316, -139248380, -148466128],
    "Min_em": [-33180983, -154948627, -48819211, -63980807, -78384629, -92468485, -105856916, -118833104, -130275827, -140664223, -148892042]
}

equitysilver_social = {
    "Scenarios": ["Baseline", "S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9"],
    "Max_hv": [100842, 130901, 90757, 80673, 70589, 60505, 50421, 40336, 30252, 20168, 10084],
    "Min_ha": [92157, None, 82941, 73725, 64510, 55294, 46078, 36862, 27647, 18431, 9215],
    "Max_st": [92157, None, 83272, 74497, 65185, 55858, 46351, 37113, 27854, 18498, 9293],
    "Min_em": [92157, 1169, 82941, 73725, 64510, 55294, 46078, 36862, 27647, 18443, 9408]
}

equitysilver_economic = {
    "Scenarios": ["Baseline", "S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9"],
    "Max_hv": [3438933, 4464020, 3095039, 2751146, 2407253, 2063360, 1719466, 1375573, 1031680, 687786, 343893],
    "Min_ha": [3142756, None, 2828481, 2514205, 2199929, 1885654, 1571378, 1257102, 942827, 628551, 314275],
    "Max_st": [3142756, None, 2839766, 2540532, 2222965, 1904896, 1580700, 1265647, 949882, 630844, 316931],
    "Min_em": [3142756, 39891, 2828481, 2514205, 2199929, 1885654, 1571378, 1257102, 942827, 628976, 320857]
}

equitysilver_species = {
    "Scenarios": ["Baseline", "S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9"],
    "Max_hv": [0.7329, 0.7577, 0.7607, 0.7514, 0.7445, 0.7401, 0.7375, 0.7275, 0.7328, 0.7365, 0.7399],
    "Min_ha": [0.7543, 0.7457, 0.7534, 0.7491, 0.7466, 0.7443, 0.7456, 0.7431, 0.7487, 0.7489, 0.7502],
    "Max_st": [0.7534, 0.7457, 0.7507, 0.7484, 0.7441, 0.7440, 0.7438, 0.7428, 0.7419, 0.7425, 0.7429],
    "Min_em": [0.7258, 0.7446, 0.7137, 0.7063, 0.6987, 0.6929, 0.6930, 0.6971, 0.7071, 0.7228, 0.7373]
}
equitysilver_old = {
    "Scenarios": ["Baseline", "S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9"],
    "Max_hv": [196384, 167126, 222922.6, 247449, 286878.4, 324882.5, 367419, 420221.3, 462754.7, 502643, 535463.1],
    "Min_ha": [303678.5, 580504.3, 334075.7, 364650.4, 393684.8, 423066.2, 451206.5, 479354.5, 505137.4, 530289.9, 556633.9],
    "Max_st": [317668.7, 580504.3, 343022.16, 366396.3, 390767.51, 417504.07, 445561.57,
                                            473603.09, 500597.34, 526910.75, 552527.14],
    "Min_em": [214539.2, 574569.2, 242967.94, 275445.01, 311282.5, 349133.9, 384958.37,
                                            422547.95, 459147.55, 498725.47, 538769.88]
}

In [8]:
# Golden Bear Data I
goldenbear_net_emission = {
    "Scenarios": ["Baseline", "S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9"],
    "Max_hv": [
        -22431846, -6207187, -20816108, -24435190, -27875150, -30827647, 
        -33795281, -37203091, -40172516, -43335597, -46257710
    ],
    "Min_ha": [
        -30666617, -49427321, -32934008, -35234780, -37362830, -39353227, 
        -41384955, -43112708, -44772679, -46464434, -48017286
    ],
    "Max_st": [
        -32702620, -49427321, -34995663, -37138002, -39157846, -41090638, 
        -42923815, -44598486, -46225607, -47539676, -48576478
    ],
    "Min_em": [
        -33380413, -49406956, -35592895, -37676137, -39670890, 
        -41552426, -43331349, -44975939, -46281647, -47578947, -48649924
    ]
}

# Golden Bear Data - Social Indicator
goldenbear_social = {
    "Scenarios": ["Baseline", "S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9"],
    "Max_hv": [
        8463, 14725, 7617, 6770, 5924, 5078, 4231, 3385, 2539, 1692, 846
    ],
    "Min_ha": [
        7787, None, 6968, 6213, 5426, 4726, 3867, 3094, 2320, 1546, 780
    ],
    "Max_st": [
        7734, None, 6961, 6187, 5414, 4640, 3867, 3093, 2320, 1546, 781
    ],
    "Min_em": [
        7734, 24, 6961, 6187, 5414, 4640, 3867, 3093, 2320, 1595, 773
    ]
}

# Golden Bear Data - Economic Indicator
goldenbear_economic = {
    "Scenarios": ["Baseline", "S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9"],
    "Max_hv": [
        288619, 502182, 259757, 230895, 202033, 173171, 144310, 115448, 86586, 57724, 28862
    ],
    "Min_ha": [
        265554, None, 237650, 211883, 185047, 161180, 131881, 105539, 79128, 52752, 26607
    ],
    "Max_st": [
        263762, None, 237385, 211009, 184633, 158257, 131881, 105504, 79128, 52752, 26652
    ],
    "Min_em": [
        263762, 850, 237385, 211009, 184633, 158257, 131881, 105504, 79128, 54409, 26376
    ]
}

# Golden Bear Data - Species Indicator
goldenbear_species = {
    "Scenarios": ["Baseline", "S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9"],
    "Max_hv": [
        0.9228, 0.8086, 0.9170, 0.9198, 0.9176, 0.9157, 0.9167, 0.9165, 0.9198, 0.9224, 0.9262
    ],
    "Min_ha": [
        0.8927, 0.9285, 0.8939, 0.8886, 0.8883, 0.8848, 0.8843, 0.8875, 0.8930, 0.8988, 0.9137
    ],
    "Max_st": [
        0.8181, 0.9285, 0.8223, 0.8284, 0.8493, 0.8639, 0.8693, 0.8780, 0.8963, 0.9092, 0.9208
    ],
    "Min_em": [
        0.8666, 0.9283, 0.8689, 0.8718, 0.8769, 0.8766, 0.8811, 0.8900, 0.8943, 0.9052, 0.9191
    ]
}

# Golden Bear Data - Old Growth Indicator
goldenbear_old = {
    "Scenarios": ["Baseline", "S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9"],
    "Max_hv": [
        26625, 14652, 31478.4, 33438, 34875, 36645.55, 39003.57, 40695.13, 42677.27, 44921.4, 52275.86
    ],
    "Min_ha": [
        27323.97, 48756.38, 28667.59, 30955.1, 33431.06, 35064.01, 37873.07, 40753.68, 43093.21, 45539.53, 47156.5
    ],
    "Max_st": [
        24711.02, 48756.02, 26270.53, 28295.18, 30210.93, 32173.28, 34377.33, 36712.57, 38970.18, 42030.01, 45547.13
    ],
    "Min_em": [
        19826.7, 48567.26, 24952.22, 23096.49, 25144.47, 27405.45, 30179.17, 33194.88, 36780.47, 39921.46, 43585.06
    ]
}


In [9]:
# Red Chris Data I
redchris_net_emission = {
    "Scenarios": ["Baseline", "S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9"],
    "Max_hv": [
        -46083355, -20553822, -50234031, -52652856, -56559028, -61565532, -64192028, 
        -68328829, -71715541, -76112747, -78934797
    ],
    "Min_ha": [
        -60129917, -81552275, -62486615, -64832416, -67151265, -69383983, 
        -71593454, -73778154, -75962420, -77708010, -79628597
    ],
    "Max_st": [
        -61347742, -81552275, -63710618, -66055037, -68411836, -70656314, -72753863, 
        -74813593, -76809883, -78787054, -80471375
    ],
    "Min_em": [
        -62345424, -81523017, -64602517, -66803615, -68914305, -70959314, -72963909, 
        -75040719, -77005884, -78896293, -80492010
    ]
}

# Red Chris Data - Social Indicator
redchris_social = {
    "Scenarios": ["Baseline", "S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9"],
    "Max_hv": [
        11852, 26423, 10667, 9482, 8296, 7111, 5926, 4741, 3555, 2370, 1185
    ],
    "Min_ha": [
        10831, None, 9748, 8665, 7582, 6499, 5449, 4345, 3262, 2172, 1089
    ],
    "Max_st": [
        10831, None, 9748, 8665, 7582, 6499, 5415, 4332, 3249, 2166, 1083
    ],
    "Min_em": [
        10831, 41, 9748, 8665, 7582, 6499, 5415, 4332, 3249, 2166, 1083
    ]
}

# Red Chris Data - Economic Indicator
redchris_economic = {
    "Scenarios": ["Baseline", "S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9"],
    "Max_hv": [
        404205, 901104, 363785, 323364, 282944, 242523, 202103, 161682, 121262, 80841, 40420
    ],
    "Min_ha": [
        369393, None, 332454, 295514, 258575, 221636, 185826, 148189, 111260, 74096, 37169
    ],
    "Max_st": [
        369393, None, 332454, 295514, 258575, 221636, 184696, 147757, 110818, 73878, 36939
    ],
    "Min_em": [
        369393, 1409, 332454, 295514, 258575, 221636, 184696, 147757, 110818, 73878, 36939
    ]
}

# Red Chris Data - Species Indicator
redchris_species = {
    "Scenarios": ["Baseline", "S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9"],
    "Max_hv": [
        0.7804, 0.7300, 0.7774, 0.7825, 0.7852, 0.7786, 0.7707, 0.7725, 0.7810, 0.7804, 0.7985
    ],
    "Min_ha": [
        0.8507, 0.8120, 0.8461, 0.8435, 0.8370, 0.8341, 0.8279, 0.8231, 0.8179, 0.8178, 0.8159
    ],
    "Max_st": [
        0.7704, 0.8120, 0.7760, 0.7818, 0.7842, 0.7849, 0.7853, 0.7894, 0.7919, 0.7934, 0.8039
    ],
    "Min_em": [
        0.7207, 0.8116, 0.7425, 0.7510, 0.7538, 0.7541, 0.7507, 0.7685, 0.7828, 0.7918, 0.8011
    ]
}

# Red Chris Data - Old Growth Indicator
redchris_old = {
    "Scenarios": ["Baseline", "S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9"],
    "Max_hv": [
        100401, 47245, 104475, 106839.5, 110990.7, 114144.5, 115217.4, 119934.8, 124733.5, 128285.5, 139178.5
    ],
    "Min_ha": [
        111298.1, 150237.5, 116224.3, 119077.2, 122514.1, 126483.9, 129808.9, 133744.85, 138646.6, 142010.7, 146688.7
    ],
    "Max_st": [
        101872.9, 150237.5, 106999.7, 112276.2, 117122.4, 121883, 126727.6, 132072.5, 137455.3, 141725.8, 146174.8
    ],
    "Min_em": [
        76629.06, 149966.1, 85185.72, 92755.32, 99524.74, 104073.4, 111344.7, 121090.3, 130605.4, 138678.6, 145028.9
    ]
}


In [10]:
datasets = {
    "equitysilver": {
        "Net Emissions (tCO2e)": equitysilver_net_emission,
        "Number of Jobs": equitysilver_social,
        "Revenue ($)": equitysilver_economic,
        "Shannon Index Value": equitysilver_species,
        "Old Growth Area (hectare)": equitysilver_old
    },
    "goldenbear": {
        "Net Emissions (tCO2e)": goldenbear_net_emission,
        "Number of Jobs": goldenbear_social,
        "Revenue ($)": goldenbear_economic,
        "Shannon Index Value": goldenbear_species,
        "Old Growth Area (hectare)": goldenbear_old
    },
    "redchris": {
        "Net Emissions (tCO2e)": redchris_net_emission,
        "Number of Jobs": redchris_social,
        "Revenue ($)": redchris_economic,
        "Shannon Index Value": redchris_species,
        "Old Growth Area (hectare)": redchris_old
    }
}

# Dropdown widget for y-labels
y_label_widget = widgets.Dropdown(
    options=list(datasets["equitysilver"].keys()),  # Default to equitysilver indicators
    description="Indicator:",
    value="Net Emissions (tCO2e)"
)

# Dropdown widget for case studies
case_study_widget = widgets.Dropdown(
    options=datasets.keys(),
    description="Case Study:",
    value="equitysilver"
)

# Update function for interactive plots
def update(case_study, y_label):
    # Update available indicators based on the selected case study
    y_label_widget.options = list(datasets[case_study].keys())
    data = datasets[case_study][y_label]
    create_grouped_bar_chart(data, y_label, case_study)

# Create interactive widget
interact(update, case_study=case_study_widget, y_label=y_label_widget)


interactive(children=(Dropdown(description='Case Study:', options=('equitysilver', 'goldenbear', 'redchris'), …

<function __main__.update(case_study, y_label)>